# Original Colab experiments (kept for reference)

This is the first, exploratory notebook. The **clean, reproducible pipeline lives in `src/` and `scripts/`** (see the README).

Fixes applied to this copy: class-name order in the reports/demo (`Closed, Open, no_yawn, yawn`), DenseNet now validates on the validation set instead of the test set. Cell outputs were cleared; re-run on Colab to regenerate them. Paths still point to Google Drive (`/content/gdrive/MyDrive/...`).


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os,random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow.keras as ks
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
dataset_path = '/content/gdrive/MyDrive/drowsiness-dataset/train/'

In [ ]:
eyes_path = '/content/gdrive/MyDrive/drowsiness-dataset/train'
fig,axes = plt.subplots(2,3,figsize=(12,8))
for ax,dirr in enumerate(['Closed','Open']):
    img_dir = os.path.join(eyes_path,dirr)
    random_imgs = random.sample(os.listdir(img_dir),3)
    for i,img in enumerate(random_imgs):
        img_path = os.path.join(img_dir,img)
        eye = plt.imread(img_path)
        axes[ax,i].imshow(eye)
        axes[ax,i].set_title('{} {}'.format(dirr,eye.shape))

In [ ]:
def eyes_data():
    directory = '/content/gdrive/MyDrive/drowsiness-dataset/train'
    labels = ['Closed','Open']
    eye_arr = []
    for clss,label in enumerate(labels):
        classs = clss
        img_dir = os.path.join(directory,label)
        for img in os.listdir(img_dir):
            img_path = os.path.join(img_dir,img)
            img_array = cv2.imread(img_path)
            resized_array = cv2.resize(img_array, (224, 224))
            eye_arr.append([resized_array, classs])
    return eye_arr

In [ ]:
dir_path = '/content/gdrive/MyDrive/drowsiness-dataset/train'
fig,axes = plt.subplots(2,3,figsize=(12,7))
for ax,dirr in enumerate(['no_yawn','yawn']):
    img_dir = os.path.join(dir_path,dirr)
    random_imgs = random.sample(os.listdir(img_dir),3)
    for i,img in enumerate(random_imgs):
        img_path = os.path.join(img_dir,img)
        eye = plt.imread(img_path)
        axes[ax,i].imshow(eye)
        axes[ax,i].set_title('{} {}'.format(dirr,eye.shape))

In [ ]:
def yawnn():
    directory = '/content/gdrive/MyDrive/drowsiness-dataset/train'
    labels = ['no_yawn','yawn']
    yawn_no_yawn = []
    for clss,label in enumerate(labels):
        classs = clss+2
        img_dir = os.path.join(directory,label)
        for img in os.listdir(img_dir):
            img_path = os.path.join(img_dir,img)
            image_array = cv2.imread(img_path)

            face_classifier = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
            face = face_classifier.detectMultiScale(image_array, scaleFactor=1.3, minNeighbors=5)
            for (x, y, w, h) in face:
                image = cv2.rectangle(image_array, (x, y), (x + w, y + h), (0, 255, 0), 2)
                face_img = image[y:y+h, x:x+w]
                resized_array = cv2.resize(face_img,(224,224))
                yawn_no_yawn.append([resized_array,classs])
    return yawn_no_yawn

In [ ]:
import cv2
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axs = axes.ravel()

img_path = '/content/gdrive/MyDrive/drowsiness-dataset/train/no_yawn/336.jpg'
img_arr = cv2.imread(img_path)
rgb_img = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)

axs[0].imshow(rgb_img)
axs[0].set_title('Original Image')

face_classifier = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
faces = face_classifier.detectMultiScale(img_arr, scaleFactor=1.3, minNeighbors=5)

if len(faces) > 0:
    for (x, y, w, h) in faces:
        cv2.rectangle(img_arr, (x, y), (x + w, y + h), (0, 255, 0), 2)
        roi_color = img_arr[y:y+h, x:x+w]
        resized_array = cv2.resize(roi_color, (224, 224))
        rgb_face = cv2.cvtColor(resized_array, cv2.COLOR_BGR2RGB)
        axs[1].imshow(rgb_face)
        axs[1].set_title('Resized Face Image')
else:
    axs[1].set_title('No Face Detected')
    axs[1].axis('off')

plt.show()

In [ ]:
arr1 = eyes_data()
arr2 = yawnn()

In [ ]:
arr1.extend(arr2)
len(arr1)

In [ ]:
X = []
y = []
for feature, label in arr1:
    X.append(feature)
    y.append(label)

In [ ]:
pp = np.array(y)
Y = ks.utils.to_categorical(pp,num_classes=4)

In [ ]:
X = np.array(X)
X.shape

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2)
x_test,x_val,y_test,y_val = train_test_split(x_test,y_test,test_size=0.5)
x_train.shape ,x_test.shape ,y_train.shape ,y_test.shape

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=30
)
test_val_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow(x_train, y_train, shuffle=True)
val_generator = test_val_datagen.flow(x_val, y_val, shuffle=False)
test_generator = test_val_datagen.flow(x_test, y_test, shuffle=False)

In [ ]:
import pandas as pd
from tqdm import tqdm, trange
import numpy as np
import os
import gc
import time
from matplotlib import pyplot as plt
from skimage.transform import resize
import seaborn as sns
from keras.applications.mobilenet import preprocess_input as preprocess_input_v1, decode_predictions as decode_predictions_v1
from keras.utils import load_img, img_to_array, to_categorical
from tensorflow.keras import layers, models, backend
from keras.models import Model, Sequential
from keras.layers import Dense, Dropout, GlobalAveragePooling2D
from keras.optimizers import Adam, AdamW, RMSprop, SGD
from keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, backend
from tensorflow.keras.preprocessing import image
tf.__version__

In [ ]:
import warnings
warnings.filterwarnings(action='once')

# CNN pre-trained

# Mobilenet

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

model_base = MobileNetV2(weights='imagenet', include_top=False)
model_base.trainable = True

x = model_base.output
x = layers.GlobalAveragePooling2D()(x)
#x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
predictions = layers.Dense(4, activation='softmax')(x)
MobileNet = models.Model(inputs=model_base.input, outputs=predictions)
callbacks_list = [
    EarlyStopping(monitor='val_accuracy', patience=4, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1)
]
MobileNet.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy']
)
MobileNet.summary()

In [ ]:
hist = MobileNet.fit( train_generator, epochs=10, validation_data=val_generator, shuffle=True,callbacks = callbacks_list,batch_size = 64).history

In [ ]:
epochsNum = 10
plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['loss'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_loss'], color='#191970', marker='h')
plt.title('Train loss VS Validation loss', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['accuracy'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_accuracy'], color='#191970', marker='h')
plt.title('Train accuracy VS Validation accuracy', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('accuracy')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

In [ ]:
test_metrics = MobileNet.evaluate(test_generator)
print(f'Test Loss: {test_metrics[0]}, Test Accuracy: {test_metrics[1]}')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
predictions =  MobileNet.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
labels_new = ["Closed", "Open", "no_yawn", "yawn"]  # FIXED: same order as the label indices
print(classification_report(true_classes, predicted_classes, target_names=labels_new))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(true_classes, predicted_classes)

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels_new)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
MobileNet.save("/content/gdrive/My Drive/mobilenet_model.h5")

**Resnet**

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

model_base = ResNet50(weights='imagenet', include_top=False)
model_base.trainable = False

x = model_base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.1)(x)
predictions = layers.Dense(4, activation='softmax')(x)
ResNet = models.Model(inputs=model_base.input, outputs=predictions)
callbacks_list = [
    EarlyStopping(monitor='val_accuracy', patience=4, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1)
]
ResNet.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy']
)
ResNet.summary()

In [ ]:
hist = ResNet.fit(  train_generator, epochs=10, validation_data=val_generator, shuffle=True,callbacks = callbacks_list,batch_size = 64).history

In [ ]:
epochsNum = 6
plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['loss'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_loss'], color='#191970', marker='h')
plt.title('Train loss VS Validation loss', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['accuracy'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_accuracy'], color='#191970', marker='h')
plt.title('Train accuracy VS Validation accuracy', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('accuracy')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

In [ ]:
test_metrics = ResNet.evaluate(test_generator)
print(f'Test Loss: {test_metrics[0]}, Test Accuracy: {test_metrics[1]}')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
predictions =  ResNet.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
labels_new = ["Closed", "Open", "no_yawn", "yawn"]  # FIXED: same order as the label indices
print(classification_report(true_classes, predicted_classes, target_names=labels_new))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(true_classes, predicted_classes)

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels_new)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
ResNet.save("/content/gdrive/My Drive/resnet2_model.h5")

# vgg 16

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

model_base = VGG16(weights='imagenet', include_top=False)
model_base.trainable = True
x = model_base.output
x = layers.GlobalAveragePooling2D()(x)
#x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
predictions = layers.Dense(4, activation='softmax')(x)
VGG16_model = models.Model(inputs=model_base.input, outputs=predictions)
callbacks_list = [
    EarlyStopping(monitor='val_accuracy', patience=4, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1)
]
VGG16_model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy']
)
VGG16_model.summary()

In [ ]:
hist = VGG16_model.fit(  train_generator, epochs=10, validation_data=val_generator, shuffle=True, callbacks = callbacks_list,batch_size = 64).history

In [ ]:
epochsNum = 10
plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['loss'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_loss'], color='#191970', marker='h')
plt.title('Train loss VS Validation loss', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['accuracy'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_accuracy'], color='#191970', marker='h')
plt.title('Train accuracy VS Validation accuracy', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('accuracy')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

In [ ]:
test_metrics = VGG16_model.evaluate(test_generator)
print(f'Test Loss: {test_metrics[0]}, Test Accuracy: {test_metrics[1]}')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
predictions =  VGG16_model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
labels_new = ["Closed", "Open", "no_yawn", "yawn"]  # FIXED: same order as the label indices
print(classification_report(true_classes, predicted_classes, target_names=labels_new))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(true_classes, predicted_classes)

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels_new)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
VGG16_model.save("/content/gdrive/My Drive/vgg16_model.h5")

**DenseNet**

In [ ]:
from tensorflow.keras.applications import DenseNet121
model_base = DenseNet121(weights='imagenet',include_top= False)
model_base.trainable = False

x = model_base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.1)(x)

predictions = layers.Dense(4,activation='softmax')(x)
DenseNet = models.Model(inputs= model_base.input, outputs=predictions)

callbacks_list = [
                     EarlyStopping(monitor='val_accuracy', patience=4, verbose=1),
                     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1)
]
DenseNet.compile(loss='categorical_crossentropy', optimizer=  tf.keras.optimizers.Adam(learning_rate=0.0001) ,metrics=['accuracy'])
DenseNet.summary()

In [ ]:
hist = DenseNet.fit(train_generator, epochs=10, validation_data=val_generator, shuffle=True,callbacks = callbacks_list,batch_size = 64).history

In [ ]:
epochsNum = 10
plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['loss'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_loss'], color='#191970', marker='h')
plt.title('Train loss VS Validation loss', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

plt.figure(figsize=(15, 7.5), dpi=40)
sns.set_style('whitegrid')
plt.plot([str(x+1) for x in range(epochsNum)], hist['accuracy'], color='#0000FF', marker='o')
plt.plot([str(x+1) for x in range(epochsNum)], hist['val_accuracy'], color='#191970', marker='h')
plt.title('Train accuracy VS Validation accuracy', fontsize=15, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('accuracy')
plt.legend(['Train', 'Validation'], loc='best')
plt.show()

In [ ]:
test_metrics = DenseNet.evaluate(test_generator)
print(f'Test Loss: {test_metrics[0]}, Test Accuracy: {test_metrics[1]}')

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
predictions =  DenseNet.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
labels_new = ["Closed", "Open", "no_yawn", "yawn"]  # FIXED: same order as the label indices
print(classification_report(true_classes, predicted_classes, target_names=labels_new))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)
conf_matrix = confusion_matrix(true_classes, predicted_classes)

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=labels_new)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
DenseNet.save("/content/gdrive/My Drive/densnet_model.h5")

In [ ]:
from tensorflow.keras.models import load_model
model = load_model("/content/gdrive/My Drive/efficient_model.h5")
model.summary()

In [ ]:
!pip install gradio

prediction

In [ ]:
from PIL import Image
import numpy as np
import cv2
import tensorflow as tf
import gradio as gr
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
model = tf.keras.models.load_model('/content/gdrive/My Drive/vgg16_model.h5')
labels_combined = ["Closed", "Open", "no_yawn", "yawn"]  # FIXED: the 4 classes are NOT combinations
IMG_SIZE = 224
def prepare_face(face_img):
    """Prétraitement de l'image pour le modèle."""
    face_img = cv2.resize(face_img, (IMG_SIZE, IMG_SIZE))
    face_img = face_img / 255.0
    face_img = np.expand_dims(face_img, axis=0)
    return face_img
def predict_image(img):
    """Effectue une prédiction sur une image."""
    img = np.array(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=1, minSize=(40, 40))

    if len(faces) == 0:
        return img, "No face detected"
    (x, y, w, h) = faces[0]
    face = img[y:y + h, x:x + w]
    roi_gray = gray[y:y + h, x:x + w]
    eyes = eye_cascade.detectMultiScale(roi_gray)
    eye_status = "Open" if len(eyes) > 0 else "Closed"
    prepared_face = prepare_face(face)
    prediction = model.predict(prepared_face)[0]
    predicted_index = np.argmax(prediction)
    predicted_label = labels_combined[predicted_index]
    predicted_probability = prediction[predicted_index]
    cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(face, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)
    result = f"Predicted Class: {predicted_label} (Probability: {predicted_probability * 100:.2f}%)"
    return img, result
description = "This system uses a CNN model to detect drowsiness signs (yawn, closed eyes, etc.) as a single class."
gr.Interface(
    fn=predict_image,
    inputs="image",
    outputs=["image", "text"],
    title="Driver Drowsiness Detection",
    description=description
).launch(debug=True)